# 07 · Inverses and the pseudoinverse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Inversas y la pseudoinversa** — Usar la pseudoinversa en sistemas reales singulares, altos y anchos, e inspeccionar su geometría de forma interactiva.

Use the pseudoinverse on real singular, tall, and wide systems, then inspect the geometry interactively.

## What you will be able to do

- Diagnose whether a real-data square matrix is invertible and explain why duplicated information makes it singular.
- Compute the Moore-Penrose pseudoinverse and verify its four defining conditions.
- Explain the difference between wide, square, and tall systems using real California housing observations.
- Solve a real 20,433-by-7 least-squares problem and interpret the residual rather than pretending an exact solution exists.
- Unfold a real image tensor, solve a wide system with the pseudoinverse, and fold the minimum-norm solution back into an image.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from sklearn.datasets import load_digits

# Enable ipywidgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

HOUSING = (
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
    "datasets/housing/housing.csv"
)
housing = pd.read_csv(HOUSING).dropna().reset_index(drop=True)

features = [
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
]

X_raw = housing[features].to_numpy(float)
feature_mean = X_raw.mean(axis=0)
feature_std = X_raw.std(axis=0)
X_scaled = (X_raw - feature_mean) / feature_std

# Bias + six standardized real features -> 7 columns.
X = np.column_stack([np.ones(len(housing)), X_scaled])
y = housing["median_house_value"].to_numpy(float)
column_names = ["bias"] + features

# Real image tensor for Exercise 3.
digits = load_digits()
digit_tensor = digits.images.astype(float)  # (1797, 8, 8)

def unfold(T, axis=0):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

print("housing rows:", len(housing))
print("housing design matrix:", X.shape)
print("digit tensor:", digit_tensor.shape)
print("interactive charts: Plotly enabled (hover, zoom, pan)")

## Why this matters

In real data work, `A⁻¹` is the exception, not the default.

A classical inverse requires a matrix that is both:

- **square**, and
- **full rank**.

Real machine-learning design matrices are usually **tall**: many observations, few features. They can also become **singular** when two columns carry duplicate or redundant information. In both situations, asking for `A⁻¹` is the wrong question.

The Moore–Penrose pseudoinverse `A⁺` is defined for rectangular and singular matrices. Its meaning depends on the geometry:

- **tall system** — usually no exact solution → `A⁺b` gives the **least-squares** solution;
- **wide system** — usually infinitely many exact solutions → `A⁺b` chooses the **minimum-norm** solution;
- **singular square system** — no ordinary inverse → `A⁺` still exists.

### How to use the folded solutions / Cómo usar las soluciones plegadas

Each exercise has a **Solution / Solución** cell that is intentionally closed. First try the `TODO`; then open the solution to compare your reasoning with a reference implementation.

> 🇪🇸 Cada ejercicio tiene una celda **Solution / Solución** cerrada a propósito. Primero intenta resolver el `TODO`; después abre la solución para comparar tu razonamiento con una implementación de referencia.

### Learning cycle: Predict → Run → Explain

Before every exercise, predict:

1. Is the matrix wide, square, or tall?
2. What is its rank?
3. Should an ordinary inverse exist?
4. If not, what should the pseudoinverse mean here?

> 🇪🇸 En trabajo real con datos, `A⁻¹` es la excepción. Una inversa ordinaria exige una matriz **cuadrada y de rango completo**. Las matrices de aprendizaje automático suelen ser **altas**, y además pueden volverse **singulares** cuando dos columnas contienen información duplicada. La pseudoinversa `A⁺` sigue existiendo. En un sistema alto entrega mínimos cuadrados; en uno ancho elige la solución exacta de norma mínima; y en una matriz cuadrada singular reemplaza una inversa que no existe.
>
> **Predice → Ejecuta → Explica:** antes de cada ejercicio decide si la matriz es ancha, cuadrada o alta; cuál debería ser su rango; si existe una inversa ordinaria; y qué debería significar la pseudoinversa.


## Exercise 1 — make a real-data matrix singular

### What are you looking at?

We take **seven real California districts** spread across the dataset and all seven columns of the standardized design matrix (bias + six real features). This gives a `7×7` square matrix.

Then we deliberately duplicate one feature column. The observations are still real; the duplication is a **teaching transformation that mimics a common feature-engineering mistake**: supplying the same information twice.

### What should happen?

If two columns are identical, they are linearly dependent. Rank drops below 7, so the matrix becomes singular and `np.linalg.inv(...)` must fail.

The pseudoinverse should still exist and satisfy the four Moore–Penrose conditions.

### What should you try?

1. Build the real `7×7` matrix from the indicated rows.
2. Check its rank.
3. Duplicate one feature column and predict the new rank **before** running.
4. Try the ordinary inverse on the singular matrix.
5. Compute `A⁺` and verify the four defining conditions.
6. Use the interactive selector to switch between the original and duplicated-feature matrices.

> 🇪🇸 Tomamos **siete distritos reales de California** distribuidos a lo largo del dataset y las siete columnas del diseño. Después duplicamos deliberadamente una columna. Los datos siguen siendo reales; la duplicación simula un error común de ingeniería de variables. Si dos columnas son iguales, el rango cae y la matriz se vuelve singular. La inversa ordinaria debe fallar, pero la pseudoinversa debe seguir existiendo. Usa el selector para comparar ambas matrices.


In [ ]:
# TODO 1
# 1. Select seven spread-out real rows:
#       row_idx = np.linspace(0, len(X) - 1, 7, dtype=int)
#       A_real = X[row_idx]
#
# 2. Print A_real.shape and np.linalg.matrix_rank(A_real).
#
# 3. Make A_singular by copying A_real and duplicating one feature column:
#       A_singular[:, 2] = A_singular[:, 1]
#    Predict its rank before printing it.
#
# 4. Try np.linalg.inv(A_singular). The error is expected.
#
# 5. Compute A_plus = np.linalg.pinv(A_singular) and verify:
#       A A+ A = A
#       A+ A A+ = A+
#       (A A+)^T = A A+
#       (A+ A)^T = A+ A


In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
row_idx = np.linspace(0, len(X) - 1, 7, dtype=int)
A_real = X[row_idx].copy()

A_singular = A_real.copy()
A_singular[:, 2] = A_singular[:, 1]  # deliberate duplicate of a REAL feature

print("original / original:", A_real.shape,
      "rank =", np.linalg.matrix_rank(A_real))
print("duplicated / duplicada:", A_singular.shape,
      "rank =", np.linalg.matrix_rank(A_singular))

try:
    np.linalg.inv(A_singular)
except np.linalg.LinAlgError as e:
    print("Expected inverse failure / Fallo esperado de la inversa:", e)

A_plus = np.linalg.pinv(A_singular)

mp_checks = [
    np.allclose(A_singular @ A_plus @ A_singular, A_singular),
    np.allclose(A_plus @ A_singular @ A_plus, A_plus),
    np.allclose((A_singular @ A_plus).T, A_singular @ A_plus),
    np.allclose((A_plus @ A_singular).T, A_plus @ A_singular),
]
print("Moore–Penrose conditions / condiciones:", mp_checks)

matrix_choice = widgets.ToggleButtons(
    options=[
        ("Original real 7×7 / Real original", "original"),
        ("Duplicated feature / Variable duplicada", "singular"),
    ],
    value="original",
    description="",
)

def inspect_matrix(choice):
    A = A_real if choice == "original" else A_singular
    rank = np.linalg.matrix_rank(A)
    cond = np.linalg.cond(A)

    print("shape / forma:", A.shape)
    print("rank / rango:", rank)
    print("condition number / número de condición:", f"{cond:.3e}")

    if rank == A.shape[0]:
        inv_error = np.linalg.norm(np.linalg.inv(A) @ A - np.eye(A.shape[0]))
        print("EN: ordinary inverse exists.")
        print("ES: la inversa ordinaria existe.")
        print("||A⁻¹A - I|| =", f"{inv_error:.3e}")
    else:
        print("EN: ordinary inverse does NOT exist; columns are dependent.")
        print("ES: la inversa ordinaria NO existe; hay columnas dependientes.")

    Ap = np.linalg.pinv(A)
    reconstruction = np.linalg.norm(A @ Ap @ A - A)
    print("pseudoinverse shape / forma de A⁺:", Ap.shape)
    print("||A A⁺ A - A|| =", f"{reconstruction:.3e}")

matrix_output = widgets.interactive_output(
    inspect_matrix,
    {"choice": matrix_choice},
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Matrix diagnostic / Diagnóstico de matriz:</b> "
            "switch one data-design decision and watch rank change. / "
            "cambia una decisión del diseño y observa cómo cambia el rango."
        ),
        matrix_choice,
        matrix_output,
    ])
)


## Exercise 2 — one real dataset, three geometries

The full California housing design matrix has shape:

`(20,433 observations, 7 columns)`

so it is **very tall**. No ordinary matrix inverse is defined for it.

The pseudoinverse solves:

`w = X⁺y`

which minimizes the total squared residual. It does **not** claim that a straight line passes exactly through all 20,433 observations.

### Why the interactive charts matter

The two full-dataset charts are now **Plotly charts**. You can:

- **hover** over a point to inspect the real and predicted house values;
- **zoom** into dense regions;
- **pan** across the distribution;
- use the toolbar to reset the view.

Then keep the same seven columns but change how many **real rows** are used:

- fewer than 7 rows → **wide** system: more unknowns than equations;
- exactly 7 rows → **square** system;
- more than 7 rows → **tall** system: more equations than unknowns.

The **Rows / Filas** slider redraws an interactive Plotly chart so you can inspect each real observation and its prediction while the geometry changes.

### What should you try?

1. Predict why `np.linalg.inv(X)` cannot be called on the full dataset.
2. Compute `w = pinv(X) @ y`.
3. Check it against `np.linalg.lstsq`.
4. Compute RMSE and inspect predicted versus actual values.
5. Hover over several districts and compare prediction error.
6. Move **Rows / Filas** through the wide → square → tall transition.

> 🇪🇸 La matriz completa tiene forma `(20.433, 7)`, por lo que es **muy alta**. No existe una inversa ordinaria para una matriz rectangular. La pseudoinversa calcula la solución de mínimos cuadrados.
>
> Las gráficas ahora son **interactivas con Plotly**: pasa el cursor sobre los puntos para ver valores reales y predichos, haz **zoom**, desplázate con **pan** y reinicia la vista desde la barra de herramientas.
>
> Con el slider **Rows / Filas** mantienes las mismas siete columnas y cambias el número de filas reales: menos de 7 produce un sistema ancho, 7 uno cuadrado y más de 7 uno alto. La gráfica se actualiza y permite inspeccionar cada observación real.


In [ ]:
# TODO 2
# 1. Explain why np.linalg.inv(X) is undefined from X.shape alone.
#
# 2. Solve the full real-data problem:
#       w = np.linalg.pinv(X) @ y
#
# 3. Compare w with:
#       np.linalg.lstsq(X, y, rcond=None)
#
# 4. Compute predictions and RMSE.
#
# 5. Because the six non-bias features were standardized, compare the
#    ABSOLUTE values of w[1:] and identify the largest standardized coefficient.
#    Describe it as an association in this dataset, NOT a causal effect.


In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
print("full X / X completa:", X.shape)
print("EN: X is rectangular, so np.linalg.inv(X) is not defined.")
print("ES: X es rectangular, por lo que np.linalg.inv(X) no está definida.")

w = np.linalg.pinv(X) @ y
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("pinv == lstsq:", np.allclose(w, w_lstsq))

pred = X @ w
residuals = pred - y
rmse = np.sqrt(np.mean(residuals**2))
print("RMSE:", f"${rmse:,.0f}")

coef_idx = int(np.argmax(np.abs(w[1:])))
print(
    "largest standardized coefficient / mayor coeficiente estandarizado:",
    features[coef_idx],
    f"{w[1:][coef_idx]:,.0f}",
)
print("EN: this is an association in this linear fit, not a causal claim.")
print("ES: es una asociación en este ajuste lineal, no una afirmación causal.")

# Interactive full-dataset scatter: hover to inspect individual real districts.
scatter_df = pd.DataFrame({
    "actual": y,
    "predicted": pred,
    "residual": residuals,
    "row": np.arange(len(y)),
    "median_income": housing["median_income"].to_numpy(float),
})

fig_scatter = px.scatter(
    scatter_df,
    x="actual",
    y="predicted",
    hover_data={
        "row": True,
        "actual": ":,.0f",
        "predicted": ":,.0f",
        "residual": ":,.0f",
        "median_income": ":.2f",
    },
    opacity=0.32,
    title="20,433 real districts / distritos reales — hover to inspect",
    labels={
        "actual": "actual / real",
        "predicted": "predicted / predicho",
    },
)
lo = float(min(y.min(), pred.min()))
hi = float(max(y.max(), pred.max()))
fig_scatter.add_trace(
    go.Scatter(
        x=[lo, hi],
        y=[lo, hi],
        mode="lines",
        name="perfect prediction / predicción perfecta",
        hoverinfo="skip",
    )
)
fig_scatter.update_layout(
    height=330,
    width=650,
    margin=dict(l=55, r=20, t=55, b=50),
)
fig_scatter.show()

# Interactive residual histogram.
residual_df = pd.DataFrame({"residual": residuals})
fig_resid = px.histogram(
    residual_df,
    x="residual",
    nbins=60,
    title="Residuals / Residuos — zoom and hover",
    labels={"residual": "prediction - actual / predicción - real"},
)
fig_resid.add_vline(x=0, line_width=1)
fig_resid.update_layout(
    height=280,
    width=650,
    margin=dict(l=55, r=20, t=55, b=50),
    yaxis_title="count / conteo",
)
fig_resid.show()

rows_slider = widgets.IntSlider(
    value=20, min=3, max=40, step=1,
    description="Rows / Filas:",
    continuous_update=False,
    style={"description_width": "90px"},
)

def explore_geometry(n_rows):
    Xn = X[:n_rows]
    yn = y[:n_rows]
    wn = np.linalg.pinv(Xn) @ yn
    predn = Xn @ wn

    residual_norm = np.linalg.norm(predn - yn)
    coef_norm = np.linalg.norm(wn)
    rank = np.linalg.matrix_rank(Xn)

    if n_rows < Xn.shape[1]:
        geometry = "WIDE / ANCHO"
        meaning_en = "Usually many exact solutions; pinv chooses minimum norm."
        meaning_es = "Normalmente hay muchas soluciones exactas; pinv elige norma mínima."
    elif n_rows == Xn.shape[1]:
        geometry = "SQUARE / CUADRADO"
        meaning_en = "An ordinary inverse exists only if rank is full."
        meaning_es = "La inversa ordinaria solo existe si el rango es completo."
    else:
        geometry = "TALL / ALTO"
        meaning_en = "Usually no exact solution; pinv gives least squares."
        meaning_es = "Normalmente no hay solución exacta; pinv da mínimos cuadrados."

    print(f"{geometry}: {Xn.shape} | rank/rango={rank}")
    print("||Xw-y|| =", f"{residual_norm:.3e}",
          "| ||w|| =", f"{coef_norm:.3e}")
    print("EN:", meaning_en)
    print("ES:", meaning_es)

    long_df = pd.DataFrame({
        "row": np.tile(np.arange(n_rows), 2),
        "value": np.concatenate([yn, predn]),
        "series": (
            ["actual / real"] * n_rows
            + ["predicted / predicho"] * n_rows
        ),
    })

    fig = px.scatter(
        long_df,
        x="row",
        y="value",
        color="series",
        hover_data={"row": True, "value": ":,.0f"},
        title=f"{geometry} — same 7 columns / mismas 7 columnas",
        labels={"row": "row / fila", "value": "median house value"},
    )
    fig.update_traces(marker={"size": 9})
    fig.update_layout(
        height=290,
        width=620,
        margin=dict(l=55, r=20, t=55, b=50),
        legend_title_text="",
    )
    fig.show()

geometry_output = widgets.interactive_output(
    explore_geometry,
    {"n_rows": rows_slider},
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Geometry explorer / Explorador geométrico:</b> "
            "move through wide → square → tall using real housing rows. "
            "Hover, zoom and pan in every chart. / "
            "recorre ancho → cuadrado → alto usando filas reales. "
            "Pasa el cursor, haz zoom y desplázate en cada gráfica."
        ),
        rows_slider,
        geometry_output,
    ])
)


## Exercise 3 — pseudoinverse after unfolding a real image tensor

Now use real handwritten-digit images.

Take the first 20 images:

`T.shape = (20, 8, 8)`

Unfolding mode 0 gives:

`M.shape = (20, 64)`

This is a **wide** matrix: 20 equations and 64 unknown pixel weights.

### A target we can interpret exactly

For each real digit image, define `b` as its **mean pixel intensity**. Because the mean of 64 pixels is a linear function, the uniform weight vector

`[1/64, 1/64, ..., 1/64]`

is one exact solution of `Mx = b`.

But a wide system has many exact solutions. The pseudoinverse chooses the one with the **smallest Euclidean norm**. We can fold that 64-value solution back to an `8×8` weight image and compare it with the uniform solution.

### Why the heatmaps are interactive

The weight maps are Plotly heatmaps. Hover over any cell to inspect its **row, column and numerical weight**. You can zoom into a region and compare how individual weights change as more digit equations are added.

### What should you try?

1. Unfold `T` to `(20, 64)`.
2. Build `b` from the real image means.
3. Solve `x = M⁺b`.
4. Confirm that both the pseudoinverse solution and the uniform solution predict `b`.
5. Compare their norms.
6. Fold `x` back to `8×8`.
7. Hover over individual weights in the heatmap.
8. Move **Digits / Dígitos** and see how the minimum-norm map changes as more real equations are added.

> 🇪🇸 Ahora usamos imágenes reales de dígitos. Veinte imágenes `8×8` forman un tensor `(20,8,8)`. Al desplegarlo obtenemos una matriz ancha `(20,64)`. Definimos `b` como la intensidad media real de cada imagen. El vector uniforme `1/64` es una solución exacta conocida, pero existen muchas. La pseudoinversa elige la solución exacta de **norma mínima**.
>
> Los mapas de pesos ahora son **heatmaps interactivos de Plotly**. Pasa el cursor sobre cualquier celda para ver su fila, columna y peso numérico; también puedes hacer zoom. Después mueve **Digits / Dígitos** para observar cómo cambia la solución cuando añadimos más ecuaciones reales.


In [ ]:
# TODO 3
# 1. T = digit_tensor[:20]
# 2. M = unfold(T, 0)               # expected shape (20, 64)
# 3. b = T.mean(axis=(1, 2))        # one real mean intensity per image
# 4. x_pinv = np.linalg.pinv(M) @ b
# 5. x_uniform = np.full(64, 1 / 64)
#
# Check:
# - M @ x_pinv reproduces b
# - M @ x_uniform reproduces b
# - ||x_pinv|| <= ||x_uniform||
#
# Finally fold:
#       x_image = x_pinv.reshape(8, 8)


In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
T = digit_tensor[:20]
M = unfold(T, 0)
b = T.mean(axis=(1, 2))

x_pinv = np.linalg.pinv(M) @ b
x_uniform = np.full(M.shape[1], 1 / M.shape[1])

print("tensor / tensor:", T.shape)
print("unfolded / desplegado:", M.shape)
print("pinv residual / residuo:", f"{np.linalg.norm(M @ x_pinv - b):.3e}")
print("uniform residual / residuo uniforme:",
      f"{np.linalg.norm(M @ x_uniform - b):.3e}")
print("||x_pinv||:", f"{np.linalg.norm(x_pinv):.6f}")
print("||x_uniform||:", f"{np.linalg.norm(x_uniform):.6f}")
print("minimum norm check / chequeo norma mínima:",
      np.linalg.norm(x_pinv) <= np.linalg.norm(x_uniform) + 1e-10)

x_image = x_pinv.reshape(8, 8)
uniform_image = x_uniform.reshape(8, 8)

# Real digit shown as interactive heatmap so the original 8x8 measurements are inspectable.
fig_digit = go.Figure(
    data=go.Heatmap(
        z=T[0],
        colorbar={"title": "pixel"},
        hovertemplate="row=%{y}<br>col=%{x}<br>pixel=%{z:.1f}<extra></extra>",
    )
)
fig_digit.update_layout(
    title="Real digit / Dígito real — original 8×8 measurements",
    height=300, width=390,
    yaxis={"autorange": "reversed", "scaleanchor": "x"},
)
fig_digit.show()

def weight_heatmap(z, title):
    fig = go.Figure(
        data=go.Heatmap(
            z=z,
            zmid=0,
            colorscale="RdBu",
            hovertemplate=(
                "row=%{y}<br>col=%{x}<br>weight/peso=%{z:.6f}<extra></extra>"
            ),
            colorbar={"title": "weight / peso"},
        )
    )
    fig.update_layout(
        title=title,
        height=300, width=390,
        yaxis={"autorange": "reversed", "scaleanchor": "x"},
    )
    return fig

weight_heatmap(
    x_image,
    "Pseudoinverse minimum-norm weights / Pesos de norma mínima",
).show()

weight_heatmap(
    uniform_image,
    "Known uniform exact solution / Solución uniforme exacta",
).show()

digits_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description="Digits / Dígitos:",
    continuous_update=False,
    style={"description_width": "100px"},
)

def explore_tensor_pinv(n_digits):
    Tn = digit_tensor[:n_digits]
    Mn = unfold(Tn, 0)
    bn = Tn.mean(axis=(1, 2))

    x_min = np.linalg.pinv(Mn) @ bn
    x_known = np.full(Mn.shape[1], 1 / Mn.shape[1])

    res_min = np.linalg.norm(Mn @ x_min - bn)
    res_known = np.linalg.norm(Mn @ x_known - bn)

    print(
        f"M: {Mn.shape} | rank/rango={np.linalg.matrix_rank(Mn)} | "
        f"pinv residual/residuo={res_min:.2e}"
    )
    print(
        f"||x_pinv||={np.linalg.norm(x_min):.6f} | "
        f"||x_uniform||={np.linalg.norm(x_known):.6f}"
    )
    print("EN: both solve the same real equations; pinv selects minimum norm.")
    print("ES: ambas resuelven las mismas ecuaciones reales; pinv elige norma mínima.")

    fig = weight_heatmap(
        x_min.reshape(8, 8),
        f"Pseudoinverse weights / Pesos pinv — {n_digits} real equations",
    )
    fig.show()

tensor_output = widgets.interactive_output(
    explore_tensor_pinv,
    {"n_digits": digits_slider},
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Tensor pseudoinverse explorer / Explorador tensorial:</b> "
            "add real digit equations, then hover/zoom on the weight map. / "
            "añade ecuaciones de dígitos reales y después pasa el cursor o haz zoom "
            "sobre el mapa de pesos."
        ),
        digits_slider,
        tensor_output,
    ])
)


## What just happened

You used the pseudoinverse for **three different reasons**, all on real observations:

1. **Singular square matrix**  
   Duplicating a real feature made two columns dependent. Rank fell and the ordinary inverse disappeared, but `A⁺` still existed and satisfied the Moore–Penrose conditions.

2. **Real California housing regression**  
   `X` had shape `(20,433, 7)`: far more equations than unknowns. There is generally no exact line through all observations, so `X⁺y` returned the **least-squares** solution. The interactive scatter and residual histogram let you hover, zoom and inspect where the approximation succeeds or fails.

3. **Real digit-image tensor**  
   Unfolding `(20,8,8)` produced a wide `(20,64)` matrix. Many exact pixel-weight solutions existed, so the pseudoinverse selected the **minimum-norm** one. Interactive heatmaps made every individual pixel weight inspectable before folding the solution back to `8×8`.

### The sentence to remember

> **Inverse asks for an exact reversible square map. Pseudoinverse asks for the best-defined solution when that ideal situation is unavailable.**

Or, by geometry:

- tall → least squares;
- wide → minimum norm;
- singular → pseudoinverse still exists.

> 🇪🇸 Usaste la pseudoinversa por **tres razones distintas** con observaciones reales: una matriz cuadrada se volvió singular al duplicar una variable; la regresión de California produjo un sistema alto que necesita mínimos cuadrados; y el tensor de dígitos produjo un sistema ancho con muchas soluciones exactas, donde `pinv` eligió la de norma mínima.
>
> Las gráficas interactivas permiten inspeccionar la matemática: en vivienda puedes pasar el cursor sobre observaciones reales, hacer zoom sobre residuos y ver cómo cambia la geometría; en el tensor puedes inspeccionar el valor exacto de cada peso.
>
> **Frase para recordar:** la inversa exige un mapa cuadrado, reversible y exacto. La pseudoinversa entrega una solución bien definida cuando esa situación ideal no existe.


---

## Time for Kahoot 🎯

**Kahoot 2 — Einsum, Distance & the Pseudoinverse** · 6 questions, about 5 minutes.

> 🇪🇸 **Einsum, distancia y la pseudoinversa** — 6 preguntas, unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-2)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_2_distance_pseudoinverse.xlsx)

Next up: **08 · Recursion with matrices and vectors** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)